# ERK-KTR Full FOV Stimulation Pipeline (RTMSequence)

This notebook runs a multi-phase ERK-KTR optogenetic experiment with full-FOV light stimulation on the Jungfrau microscope.

**Workflow overview:**
1. Initialize microscope and define channels (imaging, stimulation, optocheck)
2. Configure the image processing pipeline (segmentation, tracking, feature extraction)
3. Select FOV positions in napari and build acquisition events as RTMSequence phases
4. Validate and run the experiment
5. Post-process tracks into a single `exp_data.parquet`

In [ ]:
import os
import time
from faro.core.data_structures import (
    Channel,  # basic imaging channel (config, exposure, group)
    PowerChannel,  # imaging channel with light-source power control (adds power 0-100)
    RTMSequence,  # defines one phase of the acquisition (time plan, channels, stim, etc.)
    SegmentationMethod,
)
import faro.core.utils as utils

### Experimental Settings

**Microscope:** Jungfrau (no DMD). Change the import to use a different microscope (e.g. `Moench` for DMD-based single-cell stimulation).

**Channel types:**
- `Channel(config, exposure, group)` -- basic imaging channel
- `PowerChannel(config, exposure, group, power)` -- adds light source power control (0-100)

**What to customize below:**
- `SLEEP_BEFORE_EXPERIMENT_START_in_H` -- delay before acquisition starts (hours). Useful for scheduling overnight experiments.
- `base_path` / `experiment_name` -- output directory. All data (zarr, tracks, TIFFs) is saved under this path.
- `stim_channel` -- the stimulation light channel. Adjust `power` (intensity, 0-100) and `exposure` (ms).
- `imaging_channels` -- channels acquired every timepoint. Add/remove channels as needed; the order matters (channel 0 is used for segmentation by default).
- `optocheck_channel` -- reference channel acquired only on selected frames to verify optogenetic tool expression. Typically has a longer exposure.

In [ ]:
from faro.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup(
    "TTL_ERK"
)  # select the channel group configured in Micro-Manager

In [ ]:
## Configuration
SLEEP_BEFORE_EXPERIMENT_START_in_H = (
    0  # delay before acquisition (hours); 0 = start immediately
)

## Storage -- all output (zarr, tracks, TIFFs) goes under this directory
base_path = "E:\\Alex"
experiment_name = "test"
path = os.path.join(base_path, experiment_name)

## Stimulation channel -- light used for optogenetic activation
stim_channel = PowerChannel(
    config="CyanStim",  # Micro-Manager channel preset name
    exposure=100,  # stimulation pulse duration (ms)
    group="TTL_ERK",  # Micro-Manager channel group
    power=10,  # light source intensity (0-100)
)

## Imaging channels -- acquired at every timepoint; order matters (channel 0 is used for segmentation)
imaging_channels = (
    PowerChannel(
        config="miRFP", exposure=100, group="TTL_ERK", power=99
    ),  # nuclear marker
    PowerChannel(
        config="mScarlet3", exposure=100, group="TTL_ERK", power=99
    ),  # ERK-KTR reporter
)

## Optocheck channel -- reference channel to verify optogenetic tool expression (longer exposure)
optocheck_channel = PowerChannel(
    config="mCitrine", exposure=600, group="TTL_ERK", power=99
)

### Pipeline Setup

The `ImageProcessingPipeline` runs asynchronously during acquisition. All components are optional except `storage_path`.

**Components (customize below):**

- **Segmentation** (`segmentators`): List of `SegmentationMethod` entries. Each defines:
  - `name` -- label name in the zarr (e.g. `"labels"`)
  - `segmentation_class` -- instantiated segmentator (e.g. `CellposeV4()`)
  - `use_channel` -- which imaging channel to segment (0-indexed)
  - `save_tracked` -- if `True`, saves the tracked label mask alongside the raw segmentation

- **Stimulator** (`stimulator`): Generates the stimulation mask. Options:
  - `StimWholeFOV()` -- illuminates the entire field of view (used here)
  - `StimNothing()` -- placeholder, no stimulation
  - `StimWithPipeline` subclasses (e.g. `CenterCircle`, `StimPercentageOfCell`) -- single-cell stimulation requiring segmentation labels (needs a DMD)

- **Feature extraction** (`feature_extractor`): Extracts per-cell measurements. `FE_ErkKtr("labels")` computes cytoplasmic/nuclear ERK-KTR ratio using the `"labels"` segmentation mask.

- **Tracker** (`tracker`): Links cells across frames. `TrackerTrackpy(search_range=50)` uses trackpy; `search_range` is the max pixel displacement between frames.

- **Optocheck** (`feature_extractor_ref`): Feature extraction on reference channel frames. `OptoCheckFE(used_mask="labels")` measures optogenetic reporter intensity.

**Writer:** `OmeZarrWriter` saves imaging + stim readout into a single OME-Zarr store.

In [ ]:
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    SegmentationMethod(
        name="labels",  # label layer name in the zarr store
        segmentation_class=CellposeV4(),  # Cellpose v4 with default model
        use_channel=0,  # segment on first imaging channel (miRFP)
        save_tracked=True,  # persist tracked label masks alongside raw segmentation
    )
]

stimulator = StimWholeFOV()  # illuminate entire FOV (no DMD patterning)
feature_extractor = FE_ErkKtr("labels")  # cytoplasmic/nuclear ratio using "labels" mask
tracker = TrackerTrackpy(search_range=50)  # max 50 px displacement between frames
optocheck = OptoCheckFE(used_mask="labels")  # measure optogenetic reporter per cell

from faro.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_ref=optocheck,  # runs only on ref_frames (optocheck timepoints)
)

from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(storage_path=path)  # saves images + stim readout into OME-Zarr

### GUI

Opens a napari viewer with the Micro-Manager widget. Use this to:
- Live-view the camera feed
- Select FOV positions (used by `generate_fov_positions` in the next step) using the MDA widget

In [ ]:
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc  # point the widget at our CMMCore instance
viewer.window.add_dock_widget(mm_wdg)  # dock Micro-Manager controls in napari

### Build acquisition events

**FOV positions:** `generate_fov_positions(mic, viewer=viewer)` reads positions from the napari MDA widget. Alternatives: `filename="path/to/config.json"` to load from file, or `fake_fovs=N` for testing.

**RTMSequence parameters:**
- `time_plan` -- `{"interval": seconds, "loops": N}` defines the time-lapse schedule
- `stage_positions` -- list of FOV positions (from `generate_fov_positions`)
- `channels` -- imaging channels acquired at every timepoint
- `stim_channels` -- stimulation channel(s). Only used on frames listed in `stim_frames`
- `stim_frames` -- which frames get stimulated, **relative to this phase** (0-indexed). Accepts `range()`, `set`, or `frozenset`. Negative indices count from end (e.g. `-1` = last frame)
- `stim_exposure` -- override stim exposure per-frame. `None` uses the channel's default. Can be a single value or a list matching `stim_frames` length for ramped protocols
- `ref_channels` -- reference/optocheck channel(s). Only acquired on `ref_frames`
- `ref_frames` -- which frames get reference imaging (e.g. `[-1]` = last frame only)
- `rtm_metadata` -- experiment metadata: `phase_name`, `phase_id`, `treatment_name`

**Multi-phase experiments:** Use `phase_1 + phase_2` to concatenate. Timepoints and timing are automatically offset. Note: `stim_frames` and `ref_frames` are resolved per-phase before concatenation (e.g. `range(1, 2)` means frame 1 within that phase, not globally).

**FOV batching:** `apply_fov_batching(events, time_per_fov=2.0)` adjusts timing when the number of FOVs exceeds what can be imaged within one interval. FOVs are split into sequential batches automatically.

In [ ]:
fov_positions = utils.generate_fov_positions(mic, viewer=viewer)

# Phase 1: baseline imaging with one stim pulse
phase_1 = RTMSequence(
    time_plan={"interval": 10.0, "loops": 4},  # 4 timepoints, 10 s apart
    stage_positions=fov_positions,
    channels=imaging_channels,
    stim_channels=(stim_channel,),
    stim_frames=range(1, 2),  # stimulate on frame 1 only (0-indexed, per-phase)
    rtm_metadata={
        "phase_name": "PreDrug",
        "phase_id": 0,
        "treatment_name": "Priming Phase 1 pre Drug",
    },
)

# Phase 2: post-drug with sustained stim + optocheck on last frame
phase_2 = RTMSequence(
    time_plan={"interval": 10.0, "loops": 4},  # 4 timepoints, 10 s apart
    stage_positions=fov_positions,
    channels=imaging_channels,
    stim_channels=(stim_channel,),
    stim_frames=range(1, 3),  # stimulate on frames 1 and 2
    ref_channels=(optocheck_channel,),
    ref_frames=[-1],  # optocheck on last frame only
    rtm_metadata={
        "phase_name": "PostDrug",
        "phase_id": 1,
        "treatment_name": "Sustained Phase 2 post Drug",
    },
)

# Concatenate phases -- timepoints and timing are automatically offset
events = phase_1 + phase_2

# Split FOVs into sequential batches if too many to image within one interval
events = utils.apply_fov_batching(events, time_per_fov=2.0)

print(f"Total events: {len(events)}")
utils.events_to_dataframe(events).sort_values("timestep")

### Validate Experiment

Creates the `Controller` and validates the event list. Checks include:
- FOV positions are reachable
- Channels are configured in Micro-Manager
- Pipeline stimulator requirements match event metadata

The controller also accepts a `stim_mode` parameter (set in `run_experiment`):
- `"current"` (default) -- acquire imaging, run pipeline, then stimulate in the same timepoint
- `"previous"` -- stimulate using the mask from the previous timepoint, then acquire (lower latency)

In [ ]:
ctrl = Controller(mic, pipeline, writer=writer)
ctrl.validate_events(events)  # checks channels, positions, pipeline compatibility

### Run experiment

The optional sleep loop delays acquisition start (set `SLEEP_BEFORE_EXPERIMENT_START_in_H` above).

`mm_wdg._core_link.cleanup()` disconnects the napari live view so it does not interfere with the MDA engine during acquisition.

`ctrl.run_experiment(events, stim_mode="current")` starts the acquisition. Images are stored and pipeline runs asynchronously.

In [ ]:
# Optional: wait before starting (set SLEEP_BEFORE_EXPERIMENT_START_in_H above)
for _ in range(0, int(SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600)):
    time.sleep(1)

# Disconnect napari live view so it does not interfere with the acquisition engine
try:
    mm_wdg._core_link.cleanup()
except:
    pass

ctrl.run_experiment(events, stim_mode="current")

### Post-processing

After acquisition completes:
1. `finish_experiment()` flushes remaining pipeline tasks and closes the zarr writer
2. `generate_exp_data_from_tracks()` merges per-FOV track parquet files into a single `exp_data.parquet`
3. The napari live view is reconnected for manual inspection

In [ ]:
ctrl.finish_experiment()  # flush pipeline queue and close zarr store

utils.generate_exp_data_from_tracks(path)  # merge per-FOV tracks into exp_data.parquet

# Reconnect napari live view after acquisition is done
from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)